# 1. Carga de datos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
df_limpio = pd.read_parquet('/content/drive/MyDrive/TFM/favorita_filtrado.parquet')
df_limpio.shape

Mounted at /content/drive


(4025458, 23)

In [ ]:
df_limpio.head(10)

,id,date,store_nbr,item_nbr,unit_sales,onpromotion,city,state,type,cluster,...,dcoilwtico,type_holiday,description,transferred,year,month,day_of_week,is_weekend,es_feriado,time_idx
0,618,2013-01-02,1,122095,1.0,False,Quito,Pichincha,D,13,...,93.14,None,None,None,2013,1,2,0,0,0
13,837,2013-01-02,1,358272,6.0,False,Quito,Pichincha,D,13,...,93.14,None,None,None,2013,1,2,0,0,0
12,825,2013-01-02,1,329397,5.0,False,Quito,Pichincha,D,13,...,93.14,None,None,None,2013,1,2,0,0,0
11,771,2013-01-02,1,279157,4.0,False,Quito,Pichincha,D,13,...,93.14,None,None,None,2013,1,2,0,0,0
10,767,2013-01-02,1,271479,12.0,False,Quito,Pichincha,D,13,...,93.14,None,None,None,2013,1,2,0,0,0
9,766,2013-01-02,1,271478,2.0,False,Quito,Pichincha,D,13,...,93.14,None,None,None,2013,1,2,0,0,0
15,876,2013-01-02,1,371218,3.0,False,Quito,Pichincha,D,13,...,93.14,None,None,None,2013,1,2,0,0,0
8,719,2013-01-02,1,227728,7.0,False,Quito,Pichincha,D,13,...,93.14,None,None,None,2013,1,2,0,0,0
6,673,2013-01-02,1,179615,14.0,False,Quito,Pichincha,D,13,...,93.14,None,None,None,2013,1,2,0,0,0
5,667,2013-01-02,1,173113,2.0,False,Quito,Pichincha,D,13,...,93.14,None,None,None,2013,1,2,0,0,0


In [ ]:
#Manejar variables de calendario
df_limpio['year'] = df_limpio['date'].dt.year
df_limpio['month'] = df_limpio['date'].dt.month
df_limpio['day_of_week'] = df_limpio['date'].dt.dayofweek  # codificar días de la semana
df_limpio['is_weekend'] = df_limpio['day_of_week'].isin([5, 6]).astype(int)
df_limpio['es_feriado'] = df_limpio['type_holiday'].notnull().astype(int)

In [ ]:
df_limpio.head()

,id,date,store_nbr,item_nbr,unit_sales,onpromotion,city,state,type,cluster,...,dcoilwtico,type_holiday,description,transferred,year,month,day_of_week,is_weekend,es_feriado,time_idx
0,618,2013-01-02,1,122095,1.0,False,Quito,Pichincha,D,13,...,93.14,None,None,None,2013,1,2,0,0,0
13,837,2013-01-02,1,358272,6.0,False,Quito,Pichincha,D,13,...,93.14,None,None,None,2013,1,2,0,0,0
12,825,2013-01-02,1,329397,5.0,False,Quito,Pichincha,D,13,...,93.14,None,None,None,2013,1,2,0,0,0
11,771,2013-01-02,1,279157,4.0,False,Quito,Pichincha,D,13,...,93.14,None,None,None,2013,1,2,0,0,0
10,767,2013-01-02,1,271479,12.0,False,Quito,Pichincha,D,13,...,93.14,None,None,None,2013,1,2,0,0,0


In [ ]:
#Crear índice temporal para DeepAR y TFT
df_limpio = df_limpio.sort_values('date')
df_limpio['time_idx'] = (df_limpio['date'] - df_limpio['date'].min()).dt.days

In [ ]:
df_limpio.head()

,id,date,store_nbr,item_nbr,unit_sales,onpromotion,city,state,type,cluster,...,dcoilwtico,type_holiday,description,transferred,year,month,day_of_week,is_weekend,es_feriado,time_idx
0,618,2013-01-02,1,122095,1.0,False,Quito,Pichincha,D,13,...,93.14,None,None,None,2013,1,2,0,0,0
755,33318,2013-01-02,44,953427,36.0,False,Quito,Pichincha,A,5,...,93.14,None,None,None,2013,1,2,0,0,0
754,33295,2013-01-02,44,939210,34.0,False,Quito,Pichincha,A,5,...,93.14,None,None,None,2013,1,2,0,0,0
753,33294,2013-01-02,44,939207,94.0,False,Quito,Pichincha,A,5,...,93.14,None,None,None,2013,1,2,0,0,0
752,33293,2013-01-02,44,939131,12.0,False,Quito,Pichincha,A,5,...,93.14,None,None,None,2013,1,2,0,0,0


# 2. Partición de los datos

In [ ]:
#Considerando que trabajamos con series de tiempo, la partición no se realiza al azar sino en orden cronológico
#Se reservan los últimos 60 días para test, los 60 anteriores para validación, el resto para entrenamiento
fecha_max = df_limpio['date'].max()
test_inicio = fecha_max - pd.Timedelta(days=60)
val_inicio = test_inicio - pd.Timedelta(days=60)

train = df_limpio[df_limpio['date'] < val_inicio]
val = df_limpio[(df_limpio['date'] >= val_inicio) & (df_limpio['date'] < test_inicio)]
test = df_limpio[df_limpio['date'] >= test_inicio]

print('Train:', train['date'].min(), '→', train['date'].max(), '|', train.shape[0], 'filas')
print('Val:  ', val['date'].min(), '→', val['date'].max(), '|', val.shape[0], 'filas')
print('Test: ', test['date'].min(), '→', test['date'].max(), '|', test.shape[0], 'filas')

Train: 2013-01-02 00:00:00 → 2017-04-16 00:00:00 | 3635105 filas
Val:   2017-04-17 00:00:00 → 2017-06-15 00:00:00 | 195349 filas
Test:  2017-06-16 00:00:00 → 2017-08-15 00:00:00 | 195004 filas


In [ ]:
#Guardar
df_limpio.to_parquet('/content/drive/MyDrive/TFM/favorita_filtrado.parquet')

# 3. Revisión de serie temporal

In [ ]:
combinaciones = df_limpio[['store_nbr', 'item_nbr']].drop_duplicates()
print(len(combinaciones), 'combinaciones de tienda-producto reales')

dias_totales = (df_limpio['date'].max() - df_limpio['date'].min()).days + 1
print(dias_totales, 'días en el calendario completo')
print(len(combinaciones) * dias_totales, 'filas aproximadas si completamos el calendario diario por combinación')

4078 combinaciones de tienda-producto reales
1687 días en el calendario completo
6879586 filas aproximadas si completamos el calendario diario por combinación


Considerando que son 19 tiendas en Pichincha y existen 242 productos lácteos, el máximo de combinaciones posibles son (19x242) 4598.
Encontramos 4078 combinaciones reales, lo que significa que hay 520 combinaciones que no existieron (por tiendas que no venden ciertos productos).

In [ ]:
# Separación de datos en dos tablas
#Extraer una sola vez por fecha la información que no depende de la tienda, ni del producto
calendario = df_limpio.drop_duplicates(subset='date')[
    ['date', 'year', 'month', 'day_of_week', 'is_weekend', 'es_feriado', 'dcoilwtico']
].copy()
calendario.shape

(1679, 7)

In [ ]:
#Reviso fechas faltantes
todas_las_fechas = pd.date_range(df_limpio['date'].min(), df_limpio['date'].max(), freq='D')
fechas_en_datos = df_limpio['date'].unique()
fechas_faltantes = todas_las_fechas.difference(fechas_en_datos)
print(fechas_faltantes)

DatetimeIndex(['2013-12-25', '2014-01-01', '2014-12-25', '2015-01-01',
               '2015-12-25', '2016-01-01', '2016-12-25', '2017-01-01'],
              dtype='datetime64[ns]', freq=None)


In [ ]:
#Extraer una vez por cada tienda-producto, información que no cambia con el tiempo
atributos = df_limpio.drop_duplicates(subset=['store_nbr', 'item_nbr'])[
    ['store_nbr', 'item_nbr', 'city', 'state', 'type', 'cluster', 'family', 'class', 'perishable']
].copy()
atributos.shape

(4078, 9)

In [ ]:
atributos.head()

,store_nbr,item_nbr,city,state,type,cluster,family,class,perishable
0,1,122095,Quito,Pichincha,D,13,DAIRY,2124,1
755,44,953427,Quito,Pichincha,A,5,DAIRY,2112,1
754,44,939210,Quito,Pichincha,A,5,DAIRY,2130,1
753,44,939207,Quito,Pichincha,A,5,DAIRY,2130,1
752,44,939131,Quito,Pichincha,A,5,DAIRY,2104,1


In [ ]:
#Construir el calendario
todas_las_fechas = pd.date_range(df_limpio['date'].min(), df_limpio['date'].max(), freq='D')

#Estas se calculan en base al calendario completo (1687 días)
calendario = pd.DataFrame({'date': todas_las_fechas})
calendario['year'] = calendario['date'].dt.year
calendario['month'] = calendario['date'].dt.month
calendario['day_of_week'] = calendario['date'].dt.dayofweek
calendario['is_weekend'] = calendario['day_of_week'].isin([5, 6]).astype(int)

#Estos se traen de los datos (dependen de fuentes externas) y se rellenan
info_feriados = df_limpio.drop_duplicates(subset='date')[['date', 'es_feriado', 'dcoilwtico']]
calendario = calendario.merge(info_feriados, on='date', how='left')

calendario['es_feriado'] = calendario['es_feriado'].fillna(1).astype(int) #se confirmó que los días faltantes son feriados
calendario['dcoilwtico'] = calendario['dcoilwtico'].ffill().bfill() #ser rellena con el precio más cercano

calendario.shape

(1687, 7)

In [ ]:
calendario.head()

,date,year,month,day_of_week,is_weekend,es_feriado,dcoilwtico
0,2013-01-02,2013,1,2,0,0,93.14
1,2013-01-03,2013,1,3,0,0,92.97
2,2013-01-04,2013,1,4,0,0,93.12
3,2013-01-05,2013,1,5,1,1,93.12
4,2013-01-06,2013,1,6,1,0,93.12


In [ ]:
#Construir el esqueleto (todas las combinaciones x todos los días)
calendario['key'] = 1
atributos['key'] = 1

esqueleto = atributos.merge(calendario, on='key').drop(columns='key')
esqueleto.shape # 9 de atributos + 7 de calendario

(6879586, 16)

In [ ]:
esqueleto.head()

,store_nbr,item_nbr,city,state,type,cluster,family,class,perishable,date,year,month,day_of_week,is_weekend,es_feriado,dcoilwtico
0,1,122095,Quito,Pichincha,D,13,DAIRY,2124,1,2013-01-02,2013,1,2,0,0,93.14
1,1,122095,Quito,Pichincha,D,13,DAIRY,2124,1,2013-01-03,2013,1,3,0,0,92.97
2,1,122095,Quito,Pichincha,D,13,DAIRY,2124,1,2013-01-04,2013,1,4,0,0,93.12
3,1,122095,Quito,Pichincha,D,13,DAIRY,2124,1,2013-01-05,2013,1,5,1,1,93.12
4,1,122095,Quito,Pichincha,D,13,DAIRY,2124,1,2013-01-06,2013,1,6,1,0,93.12


In [ ]:
#Pegar ventas reales y rellenar con 0 los faltantes
ventas_agg = df_limpio.groupby(['date', 'store_nbr', 'item_nbr'], as_index=False).agg(
    unit_sales=('unit_sales', 'sum'),
    onpromotion=('onpromotion', 'max')
)

esqueleto = esqueleto.merge(ventas_agg, on=['date', 'store_nbr', 'item_nbr'], how='left')
esqueleto['unit_sales'] = esqueleto['unit_sales'].fillna(0)
esqueleto['onpromotion'] = esqueleto['onpromotion'].fillna(False)

esqueleto.shape

/tmp/ipykernel_461/3543682247.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  esqueleto['onpromotion'] = esqueleto['onpromotion'].fillna(False)


(6879586, 18)

In [ ]:
#Verificar
print('Filas totales:', esqueleto.shape[0])
print('Filas que eran ventas reales:', len(df_limpio))
print('Filas nuevas completadas con cero:', esqueleto.shape[0] - len(df_limpio))
print('Vacíos restantes en unit_sales:', esqueleto['unit_sales'].isnull().sum())

Filas totales: 6879586
Filas que eran ventas reales: 4025458
Filas nuevas completadas con cero: 2854128
Vacíos restantes en unit_sales: 0


In [ ]:
#Recalcular índice temporal sobre la tabla completa
esqueleto = esqueleto.sort_values(['store_nbr', 'item_nbr', 'date'])
esqueleto['time_idx'] = (esqueleto['date'] - esqueleto['date'].min()).dt.days

In [ ]:
#Rehacer partición train-val-test
fecha_max = esqueleto['date'].max()
test_inicio = fecha_max - pd.Timedelta(days=60)
val_inicio = test_inicio - pd.Timedelta(days=60)

train = esqueleto[esqueleto['date'] < val_inicio]
val = esqueleto[(esqueleto['date'] >= val_inicio) & (esqueleto['date'] < test_inicio)]
test = esqueleto[esqueleto['date'] >= test_inicio]

print('Train:', train['date'].min(), '→', train['date'].max(), '|', train.shape[0], 'filas')
print('Val:  ', val['date'].min(), '→', val['date'].max(), '|', val.shape[0], 'filas')
print('Test: ', test['date'].min(), '→', test['date'].max(), '|', test.shape[0], 'filas')

Train: 2013-01-02 00:00:00 → 2017-04-16 00:00:00 | 6386148 filas
Val:   2017-04-17 00:00:00 → 2017-06-15 00:00:00 | 244680 filas
Test:  2017-06-16 00:00:00 → 2017-08-15 00:00:00 | 248758 filas


In [ ]:
#Guardar tabla completa
esqueleto.to_parquet('/content/drive/MyDrive/TFM/favorita_panel_completo.parquet')